<a href="https://www.kaggle.com/code/kedhareswernaidu/moonknight?scriptVersionId=229090764" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Like Paintings

In [ ]:
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image_dataset_from_directory

def load_dataset(dataset_path, img_size=(256, 256), batch_size=32, shuffle_buffer=1000):
    dataset_path = os.path.abspath(dataset_path)  # Ensure absolute path
    if not os.path.exists(dataset_path) or not any(fname.lower().endswith(('bmp', 'gif', 'jpeg', 'jpg', 'png')) for fname in os.listdir(dataset_path)):
        raise ValueError(f"No valid images found in directory {dataset_path}. Ensure the path is correct and contains image files.")
    
    def preprocess(image, label):
        image = tf.image.resize(image, img_size) / 255.0  # Normalize to [0,1]
        return image, label
    
    dataset = image_dataset_from_directory(
        dataset_path,
        label_mode=None,  # No labels required for style transfer
        image_size=img_size,
        batch_size=batch_size
    )
    
    train_ds = (dataset
                .map(lambda img: (img / 255.0), num_parallel_calls=tf.data.AUTOTUNE)
                .shuffle(shuffle_buffer)
                .prefetch(tf.data.AUTOTUNE))
    
    return train_ds

def visualize_samples(dataset, num_samples=5):
    plt.figure(figsize=(12, 6))
    for i, image in enumerate(dataset.take(num_samples)):
        plt.subplot(1, num_samples, i+1)
        plt.imshow(image.numpy())
        plt.axis('off')
    plt.show()

# Load and visualize dataset
DATASET_PATH = "/kaggle/input/c/gan-getting-started/monet_jpg"  # Ensure dataset path is correct
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
SHUFFLE_BUFFER = 1000

try:
    train_ds = load_dataset(DATASET_PATH, IMG_SIZE, BATCH_SIZE, SHUFFLE_BUFFER)
    visualize_samples(train_ds.unbatch())
except ValueError as e:
    print(e)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ------------------------------
# Helper: ResNet Block
# ------------------------------
def resnet_block(x, filters, kernel_size=3):
    initializer = tf.random_normal_initializer(0., 0.02)
    # First convolutional layer in the block
    y = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer=initializer)(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    # Second convolutional layer in the block
    y = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer=initializer)(y)
    y = layers.BatchNormalization()(y)
    # Add skip connection
    return layers.add([x, y])

# ------------------------------
# Generator: ResNet-based (CycleGAN style)
# ------------------------------
def build_generator(input_shape=(256, 256, 3), num_resnet=6):
    initializer = tf.random_normal_initializer(0., 0.02)
    inputs = layers.Input(shape=input_shape)
    
    # c7s1-64: 7x7 Convolution, stride 1, 64 filters
    x = layers.Conv2D(64, kernel_size=7, strides=1, padding='same', kernel_initializer=initializer)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Downsampling: d128, d256 (Convolution with stride 2)
    x = layers.Conv2D(128, kernel_size=3, strides=2, padding='same', kernel_initializer=initializer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(256, kernel_size=3, strides=2, padding='same', kernel_initializer=initializer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Residual blocks: series of ResNet blocks to capture complex features
    for _ in range(num_resnet):
        x = resnet_block(x, 256)
    
    # Upsampling: Use transposed convolutions to restore original resolution
    x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, padding='same', kernel_initializer=initializer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2DTranspose(64, kernel_size=3, strides=2, padding='same', kernel_initializer=initializer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # c7s1-3: Final 7x7 Convolution to produce a 3-channel output with tanh activation
    outputs = layers.Conv2D(3, kernel_size=7, strides=1, padding='same', kernel_initializer=initializer, activation='tanh')(x)
    
    return Model(inputs, outputs, name="Generator")

# ------------------------------
# Discriminator: PatchGAN-based
# ------------------------------
def build_discriminator(input_shape=(256, 256, 3)):
    initializer = tf.random_normal_initializer(0., 0.02)
    inputs = layers.Input(shape=input_shape)
    
    # First convolution: C64 (no batch normalization in first layer)
    x = layers.Conv2D(64, kernel_size=4, strides=2, padding='same', kernel_initializer=initializer)(inputs)
    x = layers.LeakyReLU(0.2)(x)
    
    # Subsequent convolutions: C128, C256, C512
    x = layers.Conv2D(128, kernel_size=4, strides=2, padding='same', kernel_initializer=initializer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    
    x = layers.Conv2D(256, kernel_size=4, strides=2, padding='same', kernel_initializer=initializer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    
    x = layers.Conv2D(512, kernel_size=4, strides=1, padding='same', kernel_initializer=initializer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    
    # Final layer: produce a one-channel output map (PatchGAN output)
    outputs = layers.Conv2D(1, kernel_size=4, strides=1, padding='same', kernel_initializer=initializer)(x)
    
    return Model(inputs, outputs, name="Discriminator")

# ------------------------------
# Instantiate and inspect the models
# ------------------------------
generator = build_generator()
discriminator = build_discriminator()

print("Generator Summary:")
generator.summary()
print("\nDiscriminator Summary:")
discriminator.summary()

In [ ]:
import time
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------
# Global Training Parameters
# ------------------------------
EPOCHS = 100            # Total number of epochs
BATCH_SIZE = 32         # Batch size
NOISE_DIM = 100         # Dimensionality of the noise vector for the generator
LAMBDA_GP = 10.0        # Gradient penalty lambda coefficient

# ------------------------------
# Wasserstein Loss Functions
# ------------------------------
def discriminator_loss_wgan(real_output, fake_output):
    # Discriminator (critic) loss: difference between fake and real outputs
    return tf.reduce_mean(fake_output) - tf.reduce_mean(real_output)

def generator_loss_wgan(fake_output):
    # Generator loss: negative mean of critic output for fake images
    return -tf.reduce_mean(fake_output)

# ------------------------------
# Gradient Penalty Calculation for WGAN-GP
# ------------------------------
def gradient_penalty(discriminator, real_images, fake_images):
    batch_size = tf.shape(real_images)[0]
    # Interpolate between real and fake images
    alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
    interpolated = alpha * real_images + (1 - alpha) * fake_images
    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interpolated)
        # Compute critic score on the interpolated images
        interpolated_score = discriminator(interpolated, training=True)
    # Compute gradients with respect to the interpolated images
    grads = gp_tape.gradient(interpolated_score, [interpolated])[0]
    grads = tf.reshape(grads, [batch_size, -1])
    gp_norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    # Gradient penalty term
    gp = tf.reduce_mean((gp_norm - 1.0) ** 2)
    return gp

# ------------------------------
# Optimizers with Adaptive Learning Rates
# ------------------------------
generator_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=4e-4, beta_1=0.5)

# ------------------------------
# Training Step Function (WGAN-GP)
# ------------------------------
@tf.function
def train_step(real_images):
    # Train the discriminator (critic) multiple times per generator update
    for _ in range(5):
        with tf.GradientTape() as disc_tape:
            fake_images = generator(tf.random.normal([BATCH_SIZE, NOISE_DIM]), training=True)
            real_output = discriminator(real_images, training=True)
            fake_output = discriminator(fake_images, training=True)
            disc_loss = discriminator_loss_wgan(real_output, fake_output)
            gp = gradient_penalty(discriminator, real_images, fake_images)
            total_disc_loss = disc_loss + LAMBDA_GP * gp
        
        gradients_of_discriminator = disc_tape.gradient(total_disc_loss, discriminator.trainable_variables)
        discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    
    # Train the generator
    with tf.GradientTape() as gen_tape:
        fake_images = generator(tf.random.normal([BATCH_SIZE, NOISE_DIM]), training=True)
        fake_output = discriminator(fake_images, training=True)
        gen_loss = generator_loss_wgan(fake_output)
    
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    
    return gen_loss, total_disc_loss

# ------------------------------
# Helper: Generate and Save Images for Monitoring
# ------------------------------
def generate_and_save_images(model, epoch, num_examples=16):
    noise = tf.random.normal([num_examples, NOISE_DIM])
    predictions = model(noise, training=False)
    # Scale predictions from [-1,1] to [0,1]
    predictions = (predictions + 1) / 2.0
    plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(predictions[i])
        plt.axis('off')
    plt.suptitle(f"Epoch {epoch+1}")
    plt.show()

# ------------------------------
# Training Loop
# ------------------------------
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()
        gen_loss_epoch = 0.0
        disc_loss_epoch = 0.0
        batches = 0
        
        for real_images in dataset:
            gen_loss, disc_loss = train_step(real_images)
            gen_loss_epoch += gen_loss
            disc_loss_epoch += disc_loss
            batches += 1
        
        avg_gen_loss = gen_loss_epoch / batches
        avg_disc_loss = disc_loss_epoch / batches
        
        print(f"Epoch {epoch+1}, Generator Loss: {avg_gen_loss.numpy():.4f}, "
              f"Discriminator Loss: {avg_disc_loss.numpy():.4f}, Time: {time.time() - start:.2f} sec")
        
        # Generate and visualize images every 10 epochs
        if (epoch + 1) % 10 == 0:
            generate_and_save_images(generator, epoch)

# ------------------------------
# Preprocess Dataset for Training
# ------------------------------
def preprocess_for_training(image):
    # Scale image from [0,1] to [-1,1]
    return image * 2.0 - 1.0

# Assume 'train_ds' is your dataset loaded from Phase 1 (yielding images in [0,1] range).
# Since our dataset from image_dataset_from_directory (Phase 1) was loaded without labels,
# it yields images only. Update the mapping accordingly.
train_ds_for_training = train_ds.map(lambda img: preprocess_for_training(img))

# ------------------------------
# Start Training
# ------------------------------
train(train_ds_for_training, EPOCHS)